In [1]:
import os
import warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

In [2]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers

import numpy as np
import random

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

E0000 00:00:1748259465.129935   15851 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748259465.173966   15851 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1748259465.442157   15851 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748259465.442176   15851 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748259465.442179   15851 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1748259465.442180   15851 computation_placer.cc:177] computation placer already registered. Please check linka

In [3]:
#Load dataset
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()
# print(y_train_tmp.shape)
x_train = []
y_train = []
x_test = []
y_test = []
for i in range (y_train_tmp.shape[0]):
    if (0 <= y_train_tmp[i][0] <= 19):
        x_train.append(x_train_tmp[i])
        y_train.append(y_train_tmp[i])
for i in range (y_test_tmp.shape[0]):
    if (0 <= y_test_tmp[i][0] <= 19):
        x_test.append(x_test_tmp[i])
        y_test.append(y_test_tmp[i])
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

x_train = x_train / 255.0
x_test = x_test / 255.0


trainY = to_categorical(y_train, num_classes = 20)
testY = to_categorical(y_test, num_classes = 20)

In [4]:
pretrained_models = []

Xception_model = keras.applications.Xception(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
VGG16_model = keras.applications.VGG16(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
VGG19_model = keras.applications.VGG19(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
ResNet50_model = keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
ResNet50V2_model = keras.applications.ResNet50V2(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
ResNet101_model = keras.applications.ResNet101(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
ResNet101V2_model = keras.applications.ResNet101V2(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
ResNet152_model = keras.applications.ResNet152(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
ResNet152V2_model = keras.applications.ResNet152V2(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)
MobileNet_model = keras.applications.MobileNet(include_top=False, weights="imagenet", input_tensor=None, input_shape=None, pooling=None,)

#add name and model
pretrained_models.append(('Xception', Xception_model))
pretrained_models.append(('VGG16', VGG16_model))
pretrained_models.append(('VGG19', VGG19_model))
pretrained_models.append(('ResNet50', ResNet50_model))
pretrained_models.append(('ResNet50V2', ResNet50V2_model))
pretrained_models.append(('ResNet101', ResNet101_model))
pretrained_models.append(('ResNet101V2', ResNet101V2_model))
pretrained_models.append(('ResNet152', ResNet152_model))
pretrained_models.append(('ResNet152V2', ResNet152V2_model))
pretrained_models.append(('MobileNet', MobileNet_model)) 


I0000 00:00:1748259471.091291   15851 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7153 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1070, pci bus id: 0000:01:00.0, compute capability: 6.1


In [5]:

models = [] 

for name, pretrained_model in pretrained_models:
    model = keras.Sequential(
        [
            keras.Input(shape=(32, 32, 3)),
            pretrained_model,

            layers.Flatten(),

            layers.Dropout(0.4),
            layers.Dense(512),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            
            layers.Dropout(0.4),
            layers.Dense(256),
            layers.BatchNormalization(),
            layers.Activation("relu"),

            layers.Dense(20, activation='sigmoid')
        ],
        name=name
    )
    # model.summary()
    models.append(model)


In [6]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

for model in models:
    print(f"Training model: {model.name}")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),
        loss=keras.losses.CategoricalCrossentropy(from_logits=False),
        metrics=['accuracy'],
    )
    model.fit(x_train, trainY, epochs=5, callbacks=[early_stopping], batch_size=32, validation_split=0.1)

Training model: Xception
Epoch 1/5


I0000 00:00:1748259501.408746   15886 service.cc:152] XLA service 0x754c180050f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748259501.408764   15886 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce GTX 1070, Compute Capability 6.1
I0000 00:00:1748259504.866296   15886 cuda_dnn.cc:529] Loaded cuDNN version 90300


  5/282 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.0266 - loss: 3.2604       

I0000 00:00:1748259518.816148   15886 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


282/282 ━━━━━━━━━━━━━━━━━━━━ 61s 101ms/step - accuracy: 0.0548 - loss: 3.0916 - val_accuracy: 0.0660 - val_loss: 2.9126
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.0691 - loss: 2.9713 - val_accuracy: 0.0750 - val_loss: 3.3305
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.0908 - loss: 2.9081 - val_accuracy: 0.1420 - val_loss: 2.7265
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.2222 - loss: 2.5078 - val_accuracy: 0.3700 - val_loss: 1.9871
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.4504 - loss: 1.7648 - val_accuracy: 0.5760 - val_loss: 1.4132
Training model: VGG16
Epoch 1/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 37s 96ms/step - accuracy: 0.0728 - loss: 3.1846 - val_accuracy: 0.0740 - val_loss: 2.9505
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 18s 65ms/step - accuracy: 0.1174 - loss: 2.8037 - val_accuracy: 0.0650 - val_loss: 3.2226
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 18s 65ms/step - accuracy: 0.1475 - loss: 2.6244 -

In [ ]:
for model in models:
    model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 57ms/step - accuracy: 0.5809 - loss: 1.4643
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.1468 - loss: 2.5347
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.1765 - loss: 2.4516
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - accuracy: 0.5445 - loss: 1.6374
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 70ms/step - accuracy: 0.0855 - loss: 2.9099
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step - accuracy: 0.5797 - loss: 1.4302
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.2317 - loss: 2.5231
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - accuracy: 0.5106 - loss: 3.4647
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 76ms/step - accuracy: 0.0976 - loss: 3.0034
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.6551 - loss: 1.1998


: 